# Predicting Next Month's Food Prices in Rwanda's Markets and Refugee Camps

Solomon Thiong Garang Leek · Mathematics for Machine Learning · Summative: First Model Deployment

## Mission and problem
My mission is to build practical technology that closes the gap in access to the basics of a healthy life for the families most often left behind in East Africa, starting with enough food.

Rwanda hosts refugees in five camps: Mahama, Kiziba, Kigeme, Mugombwa and Nyabiheke. Many families there buy food with a monthly cash transfer from the World Food Programme (WFP). When staple prices go up, the same transfer buys less food. If the people who set the transfer only see the rise in last month's figures, families have already cut back on meals.

## Who will use this, and for what decision
The users are WFP Rwanda's cash-transfer officers, who review the monthly transfer for refugees in the camps. The decision is whether next month's planned transfer still covers a basic food basket in each camp, or needs a top-up.

**Use case:** I will predict next month's retail price per kg of staple foods in Rwandan markets and refugee camps for WFP cash-transfer officers, using the commodity, the market's province, type and distance from Kigali, the time of year and last month's prices, so that they can adjust refugees' food assistance before rising prices cut what families can eat.

**Target column:** `price`, the retail price in Rwandan francs (RWF) per kg.

## Dataset
WFP Food Prices for Rwanda, from the Humanitarian Data Exchange: https://data.humdata.org/dataset/wfp-food-prices-for-rwanda

WFP and Rwanda's Ministry of Agriculture record food prices in markets across the country every month. The file has 157,793 prices from January 2000 to August 2026, covering 109 markets and 65 items.

## Data decisions
My facilitator asked me to settle six points before modelling. This is where each one is handled:

| Point | What I did | Section |
|---|---|---|
| One meaning for the target | Only retail prices per kg of food | 2.1 |
| Price flags | Only prices flagged `actual`, meaning observed in a market | 2.2 |
| Duplicates | Checked by market, commodity and date; none found | 2.3 |
| Prices over 24 years | Rwandan francs, from January 2018 onward | 2.4 |
| At least 8 meaningful features | 8 features, each justified | 3 |
| Train/test split | By time; the most recent months are the test set | 5 |

## Setup

In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

sns.set_theme(style="whitegrid")
SEED = 42

## 1. Loading the data
Put `wfp_food_prices_rwa.csv` in the same folder as this notebook. In Colab, upload it through the Files panel. Some HDX downloads have a second header row of tags such as `#date`, so I remove it if it's there.

In [ ]:
raw = pd.read_csv("wfp_food_prices_rwa.csv")
# some HDX downloads include a second header row of tags (#date, #adm1, ...)
raw = raw[~raw["date"].astype(str).str.startswith("#")].copy()
raw["price"] = raw["price"].astype(float)
print(raw.shape)
raw.head()

In [ ]:
print(raw["date"].min(), "to", raw["date"].max())
print("missing values:", int(raw.isna().sum().sum()))
raw.nunique()

The data runs from January 2000 to August 2026 and has no missing values. Some columns repeat each other: `market_id` matches `market` and `commodity_id` matches `commodity`. The only currency is RWF.

## 2. Cleaning
### 2.1 Units and price type

In [ ]:
print(raw["pricetype"].value_counts(), "\n")
pd.crosstab(raw["category"], raw["unit"])

Prices are recorded per kilogram, per litre (oil and milk), per item (eggs and some fruit) and per sack (charcoal, the only non-food item). Only 669 prices are wholesale. A price of 500 RWF for a kilo of beans and 500 RWF for one egg are not the same kind of number, so they can't share one target.

In [ ]:
# one meaning for the target: the retail price of one kilogram of a food item
food = raw[(raw["unit"] == "KG") & (raw["pricetype"] == "Retail") & (raw["category"] != "non-food")].copy()
print(len(raw), "->", len(food))

I keep retail prices per kilogram of food, which leaves 137,910 rows. This is also what a refugee pays at the market.

### 2.2 Price flags

In [ ]:
print(food["priceflag"].value_counts(), "\n")
pd.crosstab(food["date"].str[:4], food["priceflag"]).loc["2016":]

`actual` means someone recorded the price in the market. `aggregate` and `actual,aggregate` are values WFP calculated from several readings, such as weekly prices averaged into a month. These only appear from 2020.

In [ ]:
KEY = ["market", "commodity", "date"]
observed = food[food["priceflag"] == "actual"]
derived = food[food["priceflag"] != "actual"]
both = derived.merge(observed[KEY], on=KEY)
print("derived rows:", len(derived), "| of these, also have an observed price for the same market, commodity and month:", len(both))
derived.groupby("market").size().sort_values(ascending=False).head(8)

None of the 8,403 calculated values repeat an observed price for the same market, commodity and month. They aren't duplicates, but they are averages rather than single observed prices. Most of them come from the camps.

In [ ]:
obs = observed.copy()
print(len(food), "->", len(obs))

I keep only `actual` prices so every target value is a price that was seen in a market. This costs some camp rows after 2020, but the target stays consistent.

### 2.3 Duplicates

In [ ]:
print("identical rows:", obs.duplicated().sum())
print("same market, commodity and date:", obs.duplicated(KEY).sum())

# "Potatoes" and "Potatoes (Irish)" often appear for the same market and month: are they the same item twice?
pot = obs[obs["commodity"].isin(["Potatoes", "Potatoes (Irish)"])].pivot_table(
    index=["market", "date"], columns="commodity", values="price").dropna()
ratio = pot["Potatoes"] / pot["Potatoes (Irish)"]
print(f"months with both: {len(pot)} | identical price: {(ratio == 1).mean():.1%} | median price ratio: {ratio.median():.2f}")

There are no identical rows and no repeated prices for the same market, commodity and date. "Potatoes" and "Potatoes (Irish)" appear together in the same market and month 1,677 times, so I checked whether they're the same item recorded twice. They aren't: their prices match only 3.9% of the time, and "Potatoes" is about 61% of the Irish potato price. I keep both.

### 2.4 Time window and currency

In [ ]:
obs["date"] = pd.to_datetime(obs["date"])
obs["year"] = obs["date"].dt.year
staples = ["Maize", "Beans (dry)", "Rice (local)", "Cassava flour"]
trend = (obs[obs["commodity"].isin(staples)]
         .groupby([obs["date"].dt.to_period("Q").dt.to_timestamp(), "commodity"])["price"].median().unstack())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
counts = obs.groupby("year").agg(rows=("price", "size"), markets=("market", "nunique"))
ax1.bar(counts.index, counts["rows"], color="slategray")
ax1.set(title="Observed retail prices per year", xlabel="year", ylabel="rows")
ax1b = ax1.twinx()
ax1b.plot(counts.index, counts["markets"], color="darkorange", marker="o")
ax1b.set_ylabel("markets reporting", color="darkorange")
ax1b.grid(False)
trend.plot(ax=ax2)
ax2.axvline(pd.Timestamp("2018-01-01"), color="black", linestyle="--")
ax2.set(title="Median retail price of four staples (RWF per kg, quarterly)", xlabel="", ylabel="RWF per kg")
plt.tight_layout()
plt.show()
counts.T

Two things change over the 24 years. Between 2008 and 2015, prices came from 45 to 71 markets. Collection almost stopped in 2016 (13 markets) and settled at 26 to 32 markets from 2018, including the camps.

Prices also rose a lot. Staples roughly doubled between 2018 and 2026; local rice went from about 700 to about 1,300 RWF per kg.

In [ ]:
WINDOW_START = "2018-01-01"
obs = obs[obs["date"] >= WINDOW_START].copy()
print("currencies:", obs["currency"].unique())
print("rows from 2018:", len(obs))

# drop commodities with too few prices to learn from
counts = obs["commodity"].value_counts()
print("dropped commodities:", counts[counts < 100].to_dict())
obs = obs[obs["commodity"].isin(counts[counts >= 100].index)]
print("rows:", len(obs), "| commodities:", obs["commodity"].nunique(), "| markets:", obs["market"].nunique())

I use **January 2018 onward**. That period has a steady set of markets, includes all the camps, and reflects the prices officers deal with today.

I keep prices in **Rwandan francs**. The transfer is paid and spent in RWF. `usdprice` would mix food price changes with changes in the exchange rate, and since it's the target in dollars, it would also leak the answer. I drop it.

Prices still rise within 2018 to 2026. Two features in section 3, last month's local price and last month's national median, give the model the current price level so it doesn't have to work out inflation from the year alone.

I also drop "Rice (imported, Tanzanian)", which has only 12 prices. That leaves 23,582 prices for 19 foods in 37 markets.

### 2.5 Entry errors

In [ ]:
# compare each price with the median for the same commodity in the same month across Rwanda
month_median = obs.groupby(["commodity", "date"])["price"].transform("median")
obs["ratio_to_median"] = obs["price"] / month_median
print(obs["ratio_to_median"].describe(percentiles=[0.001, 0.01, 0.99, 0.999]).round(2))
obs.loc[(obs["ratio_to_median"] > 4) | (obs["ratio_to_median"] < 0.25),
        ["date", "market", "commodity", "price", "ratio_to_median"]]

Most prices sit close to the national median for the same food in the same month: 99.8% are between 0.36 and 2.36 times the median. The 15 rows outside 0.25 to 4 times the median look like typing errors, such as imported maize flour at 7,800 RWF per kg when it cost about 500 elsewhere, or beef at 300 RWF.

In [ ]:
before = len(obs)
obs = obs[obs["ratio_to_median"].between(0.25, 4)].drop(columns="ratio_to_median")
print(before, "->", len(obs))

I remove those 15 rows.

## 3. Features
I add four features:
- `market_type`: "Refugee camp" if the market name contains "(Camp)", otherwise "Town market".
- `distance_to_kigali_km`: straight-line distance from Kigali, the main trading hub, calculated from latitude and longitude.
- `price_last_month`: the price of the same food in the same market the month before.
- `national_median_last_month`: the median price of the same food across all markets the month before.

I also take `month` from the date.

In [ ]:
data = obs.rename(columns={"admin1": "province"}).copy()
data["period"] = data["date"].dt.to_period("M")
data["month"] = data["date"].dt.month

# camp markets are named like "Mahama (Camp)"
data["market_type"] = np.where(data["market"].str.contains("(Camp)", regex=False), "Refugee camp", "Town market")

# straight-line distance from Kigali, the country's main wholesale hub (haversine formula)
KIGALI = (-1.95, 30.06)
lat, lon = np.radians(data["latitude"]), np.radians(data["longitude"])
klat, klon = np.radians(KIGALI[0]), np.radians(KIGALI[1])
a = np.sin((lat - klat) / 2) ** 2 + np.cos(lat) * np.cos(klat) * np.sin((lon - klon) / 2) ** 2
data["distance_to_kigali_km"] = (2 * 6371 * np.arcsin(np.sqrt(a))).round(1)

# last month's price of the same item in the same market
prev = data[["market", "commodity", "period", "price"]].rename(columns={"price": "price_last_month"})
prev["period"] = prev["period"] + 1
data = data.merge(prev, on=["market", "commodity", "period"], how="left")

# last month's median price of the same item across all reporting markets
national = data.groupby(["commodity", "period"])["price"].median().rename("national_median_last_month").reset_index()
national["period"] = national["period"] + 1
data = data.merge(national, on=["commodity", "period"], how="left")

data[["price_last_month", "national_median_last_month"]].isna().mean().round(3)

About 29% of rows have no price for the same market in the previous month, mostly because some markets skip months. The national median is missing for only 7%.

After filtering, some columns no longer vary:

In [ ]:
# after filtering, these columns hold a single value and carry no information
data[["unit", "pricetype", "priceflag", "currency"]].nunique()

`unit`, `pricetype`, `priceflag` and `currency` each hold one value now, so I drop them.

Six columns describe location: `admin1` (province), `admin2` (district), `market`, `market_id`, `latitude` and `longitude`.

In [ ]:
data.groupby("province").agg(districts=("admin2", "nunique"), markets=("market", "nunique"),
                             camps=("market_type", lambda s: (s == "Refugee camp").sum() > 0),
                             km_min=("distance_to_kigali_km", "min"), km_max=("distance_to_kigali_km", "max"))

After filtering, there are only 8 districts and 37 markets, and Kigali City has a single market. I keep province for regional differences and distance from Kigali for remoteness within a province. District, market name, market ID, latitude and longitude repeat the same information, or would just teach the model individual market names, so I drop them.

The 8 features I use:

| Feature | Type | Reason |
|---|---|---|
| `commodity` | categorical | Foods differ hugely in price; beef costs over 10 times as much as cabbage. |
| `province` | categorical | Supply and trade routes differ by region. |
| `market_type` | categorical | Camp or town market, which is what my user needs to compare. |
| `distance_to_kigali_km` | numeric | Remote markets can have higher transport costs. |
| `year` | numeric | The general upward trend from 2018 to 2026. |
| `month` | numeric | Harvests push prices down and lean months push them up. |
| `price_last_month` | numeric | The current price level in this market. |
| `national_median_last_month` | numeric | The current price level across Rwanda. |

Not used: `usdprice` (target in dollars), `category` (decided by commodity), `commodity_id` and `market_id` (IDs), `date` (replaced by year and month), and the constant and repeated location columns. I only keep rows that have both last-month prices.

In [ ]:
TARGET = "price"
CATEGORICAL = ["commodity", "province", "market_type"]
NUMERIC = ["distance_to_kigali_km", "year", "month", "price_last_month", "national_median_last_month"]
FEATURES = CATEGORICAL + NUMERIC

model_df = data.dropna(subset=["price_last_month", "national_median_last_month"]).reset_index(drop=True)
print(len(data), "->", len(model_df), "rows with last month's prices available")
print("features:", len(FEATURES))
model_df[["date", "market"] + FEATURES + [TARGET]].head()

That leaves 16,735 prices.

## 4. Exploring the data
### 4.1 Price by commodity

In [ ]:
order = model_df.groupby("commodity")[TARGET].median().sort_values().index
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(data=model_df, y="commodity", x=TARGET, order=order, color="lightsteelblue", fliersize=2, ax=ax)
ax.set(xscale="log", title="Retail price per kg by commodity (2018 to 2026, log scale)", xlabel="RWF per kg", ylabel="")
plt.show()

Median prices go from about 150 RWF per kg for cabbage to about 2,800 for beef, so knowing the food already explains a lot. Each box is wide because prices moved a lot over 2018 to 2026. Vegetables and tubers vary the most; salt and rice are the most stable.

### 4.2 Correlations and seasons

In [ ]:
corr = model_df[NUMERIC + [TARGET]].corr()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={"width_ratios": [1.2, 1]})
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax1)
ax1.set_title("Correlation heatmap (numeric features and price)")
season = model_df[model_df["commodity"].isin(staples)].copy()
season["vs_year_avg"] = season[TARGET] / season.groupby(["commodity", "year"])[TARGET].transform("mean")
sns.lineplot(data=season, x="month", y="vs_year_avg", hue="commodity", errorbar=None, marker="o", ax=ax2)
ax2.axhline(1, color="gray", linestyle="--")
ax2.set(title="Seasonal pattern: price relative to that year's average", xticks=range(1, 13), ylabel="price / yearly mean")
plt.tight_layout()
plt.show()
corr[TARGET].drop(TARGET).round(3)

Last month's market price (r = 0.984) and last month's national median (r = 0.983) move almost exactly with this month's price. Year has a moderate link (0.27) because prices rose over time. Distance (0.04) and month (0.00) show almost no straight-line link.

Month still matters, but in a curved way that a correlation can't pick up. Maize is about 10% below its yearly average from March to June and about 33% above it in December, before the harvest. Beans are cheapest in January and dearest in November.

The two last-month prices are strongly linked to each other (0.99). I keep both because one describes the specific market and the other the whole country, and a market that's unusually expensive tends to drift back toward the national price.

### 4.3 Camps compared with town markets

In [ ]:
camp_gap = (model_df[model_df["commodity"].isin(staples)]
            .groupby(["commodity", "market_type"])[TARGET].median().unstack())
camp_gap["camp_vs_town_%"] = ((camp_gap["Refugee camp"] / camp_gap["Town market"] - 1) * 100).round(1)
camp_gap.round(0)

For the four main staples, camp prices are within 6% of town prices. Cassava flour is actually 6% cheaper in the camps. So the risk for refugees comes from prices rising across the whole country, not from camps being unusually expensive. That's why national and last-month prices matter most in the models.

## 5. Train/test split, encoding and scaling
I split by time instead of at random. With a random 80/20 split, the model could train on March 2026 and then be tested on February 2026, so it would effectively know the future. My user always predicts forward, from known prices to next month's price.

- **Train:** February 2018 to August 2025
- **Test:** September 2025 to July 2026, the most recent months

The test set is about 7% of rows, smaller than the usual 20%, but it covers almost a full year of seasons and is the fairest test of how the model would do today.

I one-hot encode the three categorical features and standardise the five numeric ones, fitting both on the training set only.

In [ ]:
# time-based split: train on the past, test on the most recent months
TEST_START = "2025-09-01"
train = model_df[model_df["date"] < TEST_START]
test = model_df[model_df["date"] >= TEST_START]
print("train:", train["date"].min().date(), "to", train["date"].max().date(), len(train))
print("test: ", test["date"].min().date(), "to", test["date"].max().date(), len(test))

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])
X_train_s = preprocessor.fit_transform(X_train)  # fit on train only
X_test_s = preprocessor.transform(X_test)
X_train_s.shape, X_test_s.shape

That gives 15,600 training rows and 1,135 test rows, with 31 columns after encoding.

## 6. Model 1: linear regression with gradient descent (SGDRegressor)
### 6.1 Tuning
I train `SGDRegressor` one epoch at a time with `partial_fit` so I can record the loss after every epoch. For tuning, I hold out the last 12 months of the training period (September 2024 to August 2025) as a validation set, which follows the same forward-in-time idea as the test split. I tried four learning rates, four regularisation strengths and five epoch counts. Each setting is scored on the average of its last five epochs, because a single epoch can be noisy.

In [ ]:
# validation = the last 12 months of the training period, so tuning also looks forward in time
VAL_START = "2024-09-01"
is_val = (train["date"] >= VAL_START).values
X_tr, X_val = X_train_s[~is_val], X_train_s[is_val]
y_tr, y_val = y_train[~is_val], y_train[is_val]
print("tuning train:", len(y_tr), "| validation:", len(y_val))


def train_sgd(X_a, y_a, X_b, y_b, eta0, alpha, epochs):
    model = SGDRegressor(eta0=eta0, alpha=alpha, random_state=SEED)
    rng = np.random.default_rng(SEED)
    loss_a, loss_b = [], []
    for _ in range(epochs):
        idx = rng.permutation(len(X_a))  # reshuffle every epoch
        model.partial_fit(X_a[idx], y_a.values[idx])
        loss_a.append(mean_squared_error(y_a, model.predict(X_a)))
        loss_b.append(mean_squared_error(y_b, model.predict(X_b)))
    return model, loss_a, loss_b


lrs = [0.0001, 0.001, 0.01, 0.05]
alphas = [0.0001, 0.01, 0.1, 1.0]
epoch_options = [10, 20, 50, 100, 150]  # fewer than 10 epochs is too unstable to trust

curves, rows = {}, []
for lr in lrs:
    for alpha in alphas:
        _, _, val_loss = train_sgd(X_tr, y_tr, X_val, y_val, lr, alpha, 150)
        curves[(lr, alpha)] = val_loss
        for ep in epoch_options:
            # average of the last 5 epochs, because single epochs are noisy
            rows.append([lr, alpha, ep, np.mean(val_loss[ep - 5:ep])])

tuning = pd.DataFrame(rows, columns=["eta0", "alpha", "epochs", "val_mse"]).sort_values("val_mse")
tuning.head(10).style.format({"eta0": "{:g}", "alpha": "{:g}", "val_mse": "{:,.0f}"})

The best setting is a learning rate (eta0) of 0.001, alpha of 0.0001 and 50 epochs, with a validation MSE of 71,181.

### 6.2 Effect of learning rate and regularisation

In [ ]:
best_lr, best_alpha, best_epochs = tuning.iloc[0][["eta0", "alpha", "epochs"]]
best_epochs = int(best_epochs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4.8))
for lr in lrs:
    ax1.plot(curves[(lr, best_alpha)], label=f"eta0={lr}")
for alpha in alphas:
    ax2.plot(curves[(best_lr, alpha)], label=f"alpha={alpha}")
ax1.set(title=f"Learning rate (alpha={best_alpha})", xlabel="epoch", ylabel="validation MSE", yscale="log")
ax2.set(title=f"Regularisation (eta0={best_lr})", xlabel="epoch", ylabel="validation MSE", yscale="log")
ax1.legend()
ax2.legend()
plt.show()
print("best:", best_lr, best_alpha, best_epochs)

At 0.0001 the learning rate is too slow and takes about 60 epochs to catch up. At 0.01 and 0.05 the loss drops fast but jumps around, with spikes to about 260,000 at 0.05. 0.001 is both quick and steady. Alpha makes little difference up to 0.01. At 0.1 the error rises to about 84,000, and at 1.0 the weights are held back so much that it stays near 380,000.

### 6.3 Loss curve

In [ ]:
sgd, train_loss, test_loss = train_sgd(X_train_s, y_train, X_test_s, y_test, best_lr, best_alpha, best_epochs)

# epoch 0 = untrained model (all weights 0, so it predicts 0 RWF)
train_loss = [mean_squared_error(y_train, np.zeros(len(y_train)))] + train_loss
test_loss = [mean_squared_error(y_test, np.zeros(len(y_test)))] + test_loss

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for ax, start in [(ax1, 0), (ax2, 1)]:
    ax.plot(range(start, best_epochs + 1), train_loss[start:], label="train MSE")
    ax.plot(range(start, best_epochs + 1), test_loss[start:], "--", label="test MSE")
    ax.set(xlabel="epoch", ylabel="MSE (RWF squared)")
    ax.legend()
ax1.set(title="SGDRegressor loss curve", yscale="log")
ax2.set_title("Same curve from epoch 1")
plt.show()
for e in [0, 1, best_epochs]:
    print(f"epoch {e}: train {train_loss[e]:,.0f} / test {test_loss[e]:,.0f}")

Epoch 0 is the untrained model, which predicts 0 RWF for everything, so the MSE starts at about 1.2 million (train) and 4.5 million (test). After one epoch it drops to 25,406 and 73,177, and both curves are flat by about epoch 5. They end at 20,065 (train) and 57,037 (test).

The gap between the two isn't overfitting. Test prices average 1,291 RWF per kg against 740 in training, and errors in RWF grow with the price. The test loss never starts climbing again.

## 7. Models 2 to 4: LinearRegression, decision tree and random forest
I picked the tree's depth and the forest's minimum leaf size with the same validation set.

In [ ]:
lin = LinearRegression().fit(X_train_s, y_train)


def val_mse(model):
    model.fit(X_tr, y_tr)
    return mean_squared_error(y_val, model.predict(X_val))


depth_scores = {d: val_mse(DecisionTreeRegressor(max_depth=d, random_state=SEED)) for d in [4, 6, 8, 10, 12, None]}
leaf_scores = {l: val_mse(RandomForestRegressor(200, min_samples_leaf=l, random_state=SEED, n_jobs=-1)) for l in [1, 5, 10, 20]}
print("tree max_depth:", {k: round(v) for k, v in depth_scores.items()})
print("forest min_samples_leaf:", {k: round(v) for k, v in leaf_scores.items()})

best_depth = min(depth_scores, key=depth_scores.get)
best_leaf = min(leaf_scores, key=leaf_scores.get)
tree = DecisionTreeRegressor(max_depth=best_depth, random_state=SEED).fit(X_train_s, y_train)
forest = RandomForestRegressor(300, min_samples_leaf=best_leaf, random_state=SEED, n_jobs=-1).fit(X_train_s, y_train)

The tree does best at depth 8 (146,848); with no depth limit it gets worse (155,328). The forest does best with 1 sample per leaf (124,224), which is still well above SGD's 71,181 on the same validation data.

## 8. Comparing the models

In [ ]:
models = {"SGDRegressor": sgd, "LinearRegression": lin, "DecisionTree": tree, "RandomForest": forest}
results = pd.DataFrame(
    [[mean_squared_error(y_train, m.predict(X_train_s)), mean_squared_error(y_test, m.predict(X_test_s))] for m in models.values()],
    index=models.keys(), columns=["train_mse", "test_mse"],
).sort_values("test_mse")
results["test_rmse_rwf"] = np.sqrt(results["test_mse"])
best_name = results.index[0]

naive = mean_squared_error(y_test, X_test["price_last_month"])
print(f"baseline: predict the training mean -> test MSE {mean_squared_error(y_test, np.full(len(y_test), y_train.mean())):,.0f}")
print(f"baseline: predict last month's price -> test MSE {naive:,.0f}")
print("best model:", best_name)
results.round(1)

**SGDRegressor has the lowest test MSE (57,037)**, an average error of about 239 RWF per kg on test prices that average 1,291, so it's the model I save.

To put that in context, always predicting the training average gives an MSE of 3.1 million, and simply repeating last month's price gives 78,964. SGD is 28% better than that. For a typical price, repeating last month's price is about as close as the model (a median error of 6% against 8%). Where the model helps is in avoiding large misses, for example when one market is out of line with the rest of the country. Those large misses are what matter when setting a transfer.

The two linear models do best because next month's price is close to a weighted mix of last month's local and national prices, which is a straight-line relationship. SGD edges out the exact least-squares solution, probably because its regularisation and limited passes keep the weights a little smaller.

The decision tree (102,727) and random forest (115,209) fit the training data far more closely (the forest's training MSE is 1,959) but do worse on the test months. Trees can't predict above the highest prices they've seen, and 2025 to 2026 prices are the highest in the data: 79 test prices are above the top 1% of training prices. A random split would have hidden this, which is why splitting by time matters here.

### What the models rely on

In [ ]:
names = [n.split("__")[1] for n in preprocessor.get_feature_names_out()]
coefs = pd.Series(lin.coef_, index=names)
coefs = coefs.reindex(coefs.abs().sort_values().index).tail(15)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))
coefs.plot.barh(ax=ax1, color=np.where(coefs > 0, "steelblue", "tomato"))
ax1.set_title("Linear regression: 15 largest coefficients (RWF per standardised unit)")
pd.Series(forest.feature_importances_, index=names).nlargest(10).sort_values().plot.barh(ax=ax2, color="seagreen")
ax2.set_title("Random forest feature importance")
plt.tight_layout()
plt.show()

Both models lean mostly on the two last-month prices; the random forest gives 82% of its importance to the national median. In the linear model, beef and dried fish push the prediction up, and vegetables and tubers pull it down. Province, market type, distance and month have small effects, which fits what section 4.3 showed: camp prices follow national prices.

## 9. Strongest predictor: before and after fitting a line
### Before

In [ ]:
top = train[NUMERIC].corrwith(train[TARGET]).abs().idxmax()
print(top)
plt.figure(figsize=(8, 5))
plt.scatter(X_train[top], y_train, alpha=0.2, s=8)
plt.xlabel(f"{top} (RWF per kg)")
plt.ylabel("price (RWF per kg)")
plt.title("Before fitting")
plt.show()

### After
The same points with a line from a one-feature linear regression, fitted only for this plot.

In [ ]:
line = LinearRegression().fit(X_train[[top]], y_train)
xs = pd.DataFrame({top: np.linspace(0, X_train[top].max(), 50)})

plt.figure(figsize=(8, 5))
plt.scatter(X_train[top], y_train, alpha=0.2, s=8)
plt.plot(xs[top], line.predict(xs), color="red", linewidth=2.5,
         label=f"y = {line.coef_[0]:.3f}x + {line.intercept_:.1f}")
plt.xlabel(f"{top} (RWF per kg)")
plt.ylabel("price (RWF per kg)")
plt.title("After fitting")
plt.legend()
plt.show()
print(f"one-feature test MSE: {mean_squared_error(y_test, line.predict(X_test[[top]])):,.0f}")

With a slope of 0.996 and an intercept of 8.7, last month's price on its own predicts almost the same price this month. This one-feature line scores 78,971 on the test set, the same as just repeating last month's price. The full model gets to 57,037 by adding the commodity, location, season and national price. The lone point far to the right (9,350 last month, about 700 this month) is probably a typing error in the previous month that my filter missed.

## 10. Saving the model
I save the encoder, the scaler and the SGD model together as one `Pipeline`, so the API can take raw values. I also save the training data, because the API's `/retrain` endpoint adds new prices to it. The lists of foods and provinces and the ranges of the numeric features printed below are the values the API accepts.

In [ ]:
os.makedirs("../API", exist_ok=True)  # the API folder sits next to this notebook's folder
best_model = Pipeline([("preprocess", preprocessor), ("model", models[best_name])])
joblib.dump(best_model, "../API/best_model.pkl")

# the API's /retrain endpoint uses this file together with uploaded rows
model_df[FEATURES + [TARGET]].to_csv("../API/training_data.csv", index=False)

# allowed values for the API and the app
print("commodities:", sorted(model_df["commodity"].unique()))
print("provinces:", sorted(model_df["province"].unique()))
model_df[NUMERIC].describe().loc[["min", "max"]]

## 11. Predicting one test row
This function loads the saved pipeline and predicts the price for one row from the test set, here a refugee camp market.

In [ ]:
def predict_one(row, path="../API/best_model.pkl"):
    model = joblib.load(path)
    return model.predict(row[FEATURES])[0]


camp_rows = test[test["market_type"] == "Refugee camp"]
example = camp_rows.iloc[[0]] if len(camp_rows) else test.iloc[[0]]
print(example["market"].iloc[0], example["date"].iloc[0].date())
display(example[FEATURES])
print("predicted:", round(predict_one(example), 1), "RWF per kg")
print("actual:   ", example[TARGET].iloc[0], "RWF per kg")

For cassava in Nyabiheke camp in September 2025, last month's price was 362.5 RWF per kg and the national median was 350. The model predicts 389.5 RWF per kg; the actual price was 375, so the error is about 4%. For an officer, this reads as a small rise to plan the next transfer around.

## 12. Summary and limitations
Using 8 features and 16,735 observed retail prices from 2018 to 2026, SGDRegressor predicts next month's price per kg with a test MSE of 57,037 (about 239 RWF per kg) on the most recent 11 months. That is 28% lower than assuming prices don't change. Last month's local and national prices carry most of the signal, and the food and the season adjust them. Camp prices follow national prices closely, so national prices are the best early warning for refugees' food budgets.

Limitations:
- Prices are nominal RWF. The model follows inflation through last month's prices instead of removing it.
- A typical error of about 8% is enough to flag a rising price but not to set an exact transfer amount.
- Month is treated as a straight number in the linear models, so they only partly capture the seasonal curve.
- Rows without last month's price are dropped, so markets that report irregularly are under-represented.
- WFP's calculated camp prices after 2020 are left out to keep the target consistent.

Next, I would encode seasons properly (categories or sine and cosine terms), add fuel prices and rainfall, and retrain each month through `/retrain` as WFP publishes new prices.